# L05 · 旋转即数字 — 复数的几何本质

**目标**：复数 `a+bi` 有实部/虚部，也能写成"模 + 相位"(极坐标)。

**为什么对 Aurora 重要**：FFT 每个频点输出一个复数——**模 = 该频率有多强(响度)，相位 = 时间对齐信息**。

## 本课剧情：给二维箭头办身份证

复数 `a+bi` 就是平面上的点：实部是横坐标，虚部是纵坐标。模（`abs(z)`）是点到原点的距离，相位（`np.angle(z)`）是与 x 轴的夹角。这节课实现 `magnitude_phase(z)`，把复数拆成这两个量，对应 FFT 每个频点的响度和时间对齐信息。

## 1. 复数的基本操作（Python 用 `j` 表示虚数单位）

`a + bj` 有两种等价描述：直角坐标 `(a, b)` 和极坐标 `(r, θ)`。转换关系：`r = √(a²+b²)`，`θ = arctan2(b, a)`；逆变换：`a = r·cos(θ)`，`b = r·sin(θ)`。

极坐标在频谱分析里比直角坐标更自然：模 `r` 直接对应"这个频率有多响"，相位 `θ` 直接对应"这个正弦波的起始时刻"。FFT 每个频点输出一个复数，取模得到 magnitude spectrogram（各频率的响度），取相位得到时间对齐信息——相位重建是声码器的核心操作。两者相互独立：改变响度不影响时序，改变起始时刻不影响响度。

## 实验入口：角度、坐标和旋转

角度 θ 对应的单位复数是 `cos(θ) + sin(θ)j`：实部是水平分量，虚部是垂直分量，模恒为 1。下面遍历一圈，验证极坐标与直角坐标的互换关系。

In [ ]:
import numpy as np
z = 3 + 4j
print('实部:', z.real, ' 虚部:', z.imag)
print('模 |z| =', abs(z), ' (=√(3²+4²)=5)')
print('相位(弧度) =', np.angle(z))

## 动手观察：复数就是"旋转的位置"

观察下表的 `real` 和 `imag` 列：角度每增加 π/4，坐标在单位圆上逆时针移一步。`magnitude` 列始终为 1，改变的只有 `phase`。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：旋转一整圈

从 0 到 2π 均匀取 8 个角度，打印每个点的实部、虚部、模和相位，验证单位圆上模始终为 1、相位就是角度本身。

In [ ]:
import numpy as np

for k in range(9):
    theta = 2 * np.pi * k / 8
    z = np.exp(1j * theta)
    print(f'k={k} theta={theta:.2f} -> ({z.real:+.3f}, {z.imag:+.3f})')


## 2. ✏️ 实现 `magnitude_phase(z)` 返回 `(模, 相位)`

**推理路线**：
1. 模是复数到原点的距离：`r = √(real² + imag²)`，Python 内置 `abs(z)` 直接返回此值，对标量和 numpy 数组均适用。
2. 相位是与 x 轴正半轴的夹角：`θ = arctan2(imag, real)`，`np.angle(z)` 封装了此计算，输出范围 `(-π, π]`，单位弧度。
3. 返回元组 `(magnitude, phase)` 是因为 FFT 后处理通常同时需要两者，比分开调用两次更高效。

**参考输入输出**：`z = 3 + 4j` → 模 = 5.0，相位 ≈ 0.9273 弧度（约 53°，即 arctan(4/3)）

<details><summary>点击查看参考实现</summary>

```python
def magnitude_phase(z):
    return abs(z), np.angle(z)
```

</details>

### 写代码前，先把变量表补完整

写 `magnitude_phase` 前明确三件事：
- 输入：复数 `z`（Python 或 numpy 复数均可）
- 关键步骤：`abs(z)` 求模；`np.angle(z)` 求相位，范围 (−π, π]，单位弧度
- 返回：元组 `(magnitude, phase)`，两者均为浮点数

In [ ]:
def magnitude_phase(z):
    # ✏️ TODO: 返回 (模, 相位) 元组
    return None  # ← 改这里


In [ ]:
import numpy as np
mag, ph = magnitude_phase(3 + 4j)
print('模 =', mag, ' 相位 =', round(ph, 4))
assert abs(mag - 5.0) < 1e-9
assert abs(ph - np.arctan2(4, 3)) < 1e-9
print('✅ 通过：你能从复数读出模和相位了。')

**🔗 Aurora 连接**：`magnitude_spectrogram` 取的就是 FFT 复数结果的模；相位在声码器/重建里很关键。下一课：把复数和三角连起来——欧拉公式。

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | 半径={radius:.2f}')
print('所有点半径都接近 1，说明它们都在单位圆上。')


## 参数实验：单位圆上 8 点的模与相位

构造 `z = np.exp(1j * np.linspace(0, 2*np.pi, 8))`，打印每个点的模和相位，确认模恒为 1.0，相位均匀分布在 0 到约 5.5 弧度（即 0, π/4, π/2, …, 7π/4）。

再把 `3 + 4j` 换成 `3 - 4j`，观察相位从 +0.9273 变为 -0.9273：虚部取反即相位取反，模不变——验证模与相位相互独立。

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | 半径={radius:.2f}')
print('所有点半径都接近 1，说明它们都在单位圆上。')


## 本课收束

现在能用 `magnitude_phase(z)` 把任意复数拆成模（`abs(z)`）和相位（`np.angle(z)`），两者均为浮点数。Aurora 的 `magnitude_spectrogram` 正是对 FFT 每个频点的复数取模，得到各频率的响度谱。下一课欧拉公式 `e^{jθ} = cos(θ) + j sin(θ)` 会把极坐标表示和正弦波直接连起来。

In [ ]:
# 小检查：同一个角度，同时看 cos、sin、复数
import numpy as np

for theta in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} -> cos={z.real:+.1f}, sin={z.imag:+.1f}, complex={z.real:+.1f}{z.imag:+.1f}j')
